# Chapter 6 — How to Evaluate a Hallucination Detector

**Book alignment:** Hallucination From First Principles, Chapter 6

**Question this notebook isolates:** Can two score distributions with good AUC still fail a strict false-acceptance budget once the threshold is frozen on a calibration split, and does prevalence collapse alert precision?

Synthetic embeddings below demonstrate the geometry type only and do not reproduce the book's empirical runs.

In [ ]:
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)
print("numpy", np.__version__, "seed", SEED)

## 1 — Ranking quality: good AUC from separated distributions

Fixed orientation for this book: positive = unsupported, higher score = riskier, `FAR = P(accept | unsupported)`, `SAR = P(accept | supported)`. Synthetic energies mimic the chapter's regime (supported mean ~0.37, hard-negative mean ~0.69).

In [ ]:
def synth_energies(n_sup, n_unsup, seed):
    r = np.random.default_rng(seed)
    sup = np.clip(r.normal(0.3714, 0.16, n_sup), 0.0, 1.0)
    uns = np.clip(r.normal(0.6950, 0.17, n_unsup), 0.0, 1.0)
    return sup, uns


def auc_higher_is_riskier(sup, uns):
    # P(s_u > s_s) + 0.5 P(tie) via rank method (no sklearn needed).
    s = np.concatenate([sup, uns])
    y = np.concatenate([np.zeros_like(sup), np.ones_like(uns)])
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s))
    ranks[order] = np.arange(1, len(s) + 1)
    n0, n1 = len(sup), len(uns)
    rank_sum_pos = ranks[y == 1].sum()
    return float((rank_sum_pos - n1 * (n1 + 1) / 2) / (n0 * n1))


cal_sup, cal_uns = synth_energies(1000, 1000, 100)
eval_sup, eval_uns = synth_energies(2000, 2000, 200)
auc = auc_higher_is_riskier(eval_sup, eval_uns)
print(f"supported mean={eval_sup.mean():.4f} unsupported mean={eval_uns.mean():.4f}")
print(f"gap={float(eval_uns.mean() - eval_sup.mean()):.4f} AUC={auc:.4f}")

In [ ]:
assert (eval_uns.mean() - eval_sup.mean()) > 0.20
assert auc > 0.75, auc
print("Ranking signal is good: AUC well above 0.5.")

## 2 — Decision quality: a frozen strict-FAR threshold destroys coverage

Freeze the threshold on the calibration split (1st percentile of unsupported calibration energies, targeting FAR <= 1%), then report held-out FAR and SAR. Means separate; the low-energy tail still overlaps.

In [ ]:
tau = float(np.quantile(cal_uns, 0.01))
cal_far = float(np.mean(cal_uns <= tau))
far = float(np.mean(eval_uns <= tau))
sar = float(np.mean(eval_sup <= tau))
frr = 1.0 - sar
detect = 1.0 - far
print(f"frozen tau={tau:.4f} (from calibration split only)")
print(f"calibration FAR={cal_far:.4f} held-out FAR={far:.4f}")
print(f"held-out SAR={sar:.4f} FRR={frr:.4f} detection={detect:.4f}")

In [ ]:
assert abs(cal_far - 0.01) < 1e-9
assert far <= 0.05, far  # strict budget approximately met
assert sar < 0.50, sar  # but supported coverage collapses
print("Mean separation is not an operating point: strict FAR admits few supported examples.")

## 3 — Deployment utility: prevalence controls what alerts mean

Alert = detector flags (rejects). With `p = P(unsupported)`, alert precision is `(1-FAR)*p / ((1-FAR)*p + FRR*(1-p))`. Ranking is prevalence-free; alert value is not.

In [ ]:
def alert_precision(p, far, frr):
    det = 1.0 - far
    return (det * p) / (det * p + frr * (1 - p))


for p in (0.50, 0.05, 0.01):
    print(f"prevalence={p:.2f} alert_precision={alert_precision(p, far, frr):.4f}")
ppv_50 = alert_precision(0.50, far, frr)
ppv_05 = alert_precision(0.05, far, frr)
ppv_01 = alert_precision(0.01, far, frr)

In [ ]:
assert ppv_50 > ppv_05 > ppv_01
assert ppv_01 < 0.30, ppv_01
print("At 1% prevalence nearly every alert is a false alarm despite good AUC.")

## What we earned

Ranking (AUC), decision (frozen-threshold FAR/SAR), and deployment utility (prevalence-conditioned precision) are three different claims. Good separation survived; the strict operating point and the low-prevalence alert stream did not.

Chapter 7 keeps this evaluation discipline and deliberately searches for the valid examples that exploit what the sensor does not observe.